In [36]:
import pandas as pd
import numpy as np

import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer


It is A Regression problem where we need to find the strength column value 

In [37]:
df = pd.read_csv("concrete_data.csv")

In [38]:
df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [39]:
df.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

Checking wheather the data has 0 or -ve values because then we can't apply box - cox transformation

In [40]:
df.describe()

#Check the min row so that you cna find the 0 or -ve values

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [41]:
X_train,X_test,y_train,y_test = train_test_split(
    df.iloc[:,:-1],
    df["Strength"],
    random_state=42,
    test_size=0.2
)

In [42]:
lr = LinearRegression()

lr.fit(X_train,y_train)

y_pred = lr.predict(X_test)

r2_score(y_test,y_pred)

0.6275531792314848

In [43]:
lr = LinearRegression()


X = df.iloc[:,:-1]
y = df["Strength"]

np.mean(cross_val_score(lr,X,y,scoring='r2'))

np.float64(0.4609940491662864)

Applying Box-Cox

In [44]:
pt = PowerTransformer(method='box-cox')

#We added 0.00001 because we have zeros in the data set to make then positive to use in box cox we did it it is trick
X_train_transformed = pt.fit_transform(X_train+0.000001)
X_test_transformed = pt.transform(X_test+0.000001)

pd.DataFrame({'cols':X_train.columns,'box_cox_lamdas':pt.lambdas_})

,cols,box_cox_lamdas
0,Cement,0.177025
1,Blast Furnace Slag,0.025093
2,Fly Ash,-0.038970
3,Water,0.772682
4,Superplasticizer,0.098811
5,Coarse Aggregate,1.129813
6,Fine Aggregate,1.782018
7,Age,0.066631


In [45]:
# Suppose if we take a row in the cement columsna nd it value is 540 then we apply (540)^0.177025 to it similary we do it for the values in the cement column

In [46]:
lr = LinearRegression()
lr.fit(X_train_transformed,y_train)

y_pred2 = lr.predict(X_test_transformed)

r2_score(y_test,y_pred2)


0.8047825008078886

In [47]:
pt = PowerTransformer(method='box-cox')

X_train_transformed = pt.fit_transform(X+0.00001)

lr = LinearRegression()
np.mean(cross_val_score(lr,X_train_transformed,y,scoring='r2'))

np.float64(0.6668489649183531)

Applying Yeo-Johnson

In [48]:
pt1 = PowerTransformer()

X_train_transformed = pt1.fit_transform(X_train)
X_test_transformed = pt1.transform(X_test)

pd.DataFrame({'cols':X_train.columns,'yeo_johnson_lamdas':pt1.lambdas_})

,cols,yeo_johnson_lamdas
0,Cement,0.174348
1,Blast Furnace Slag,0.015715
2,Fly Ash,-0.161447
3,Water,0.771307
4,Superplasticizer,0.253935
5,Coarse Aggregate,1.130050
6,Fine Aggregate,1.783100
7,Age,0.019885


In [49]:
lr = LinearRegression()
lr.fit(X_train_transformed,y_train)

y_pred2 = lr.predict(X_test_transformed)

r2_score(y_test,y_pred2)

0.8161906510869553

In [50]:
pt1 = PowerTransformer()

X_train_transformed = pt1.fit_transform(X)

lr = LinearRegression()
np.mean(cross_val_score(lr,X_train_transformed,y,scoring='r2'))

np.float64(0.683462515116388)

In [51]:
pd.DataFrame({'cols':X_train.columns,'box_cox_lamdas':pt.lambdas_,'yeo_johnson_lamdas':pt1.lambdas_})

,cols,box_cox_lamdas,yeo_johnson_lamdas
0,Cement,0.172271,0.169544
1,Blast Furnace Slag,0.028052,0.016633
2,Fly Ash,-0.037072,-0.136480
3,Water,0.809568,0.808438
4,Superplasticizer,0.114979,0.264160
5,Coarse Aggregate,1.129168,1.129395
6,Fine Aggregate,1.829625,1.830764
7,Age,0.048975,0.001771
